In [ ]:
import cupy as cp
import numpy as np
import xraylib
import random
import cupyx.scipy.ndimage as ndimage
from types import SimpleNamespace
import tifffile
from utils import *
from rec import Rec

%matplotlib inline

In [ ]:
np.random.seed(10)
cp.random.seed(10)

In [ ]:
n = 1024    
ntheta = 1
ndist = 4

# P23 setup
detector_pixelsize = 3.25e-6
energy = 30  # [keV] xray energy
wavelength = 1.24e-09/energy  # [m] wave length
focusToDetectorDistance = 1.141  # [m]
z1 = np.array([0.00532, 0.00542, 0.00622, 0.0101])
z2 = focusToDetectorDistance-z1
distances = z1 * z2 / focusToDetectorDistance
magnifications = focusToDetectorDistance / z1
norm_magnifications = magnifications / magnifications[0]
voxelsize = np.abs(detector_pixelsize / magnifications[0])
print(voxelsize)

### Adjust distances for the cone beam: sample - detector, probe - sample

## Reconstructed object size

In [ ]:
nobj = 1536
print(f'Reconstructed object size {nobj}x{nobj}')

## Form Siemens star

In [ ]:
def rotate_points(pts, theta, center):
    """Rotate Nx2 points by theta (rad) about center (2,)."""
    c, s = np.cos(theta), np.sin(theta)
    R = np.array([[c, -s],
                  [s,  c]], dtype=np.float32)
    return (pts - center) @ R.T + center

def triangle_mask(X, Y, tri):
    """
    Vectorized point-in-triangle test.
    X, Y: grids (H,W)
    tri: (3,2) array of triangle vertices (x,y) in pixel coords
    Returns boolean mask (H,W).
    """
    x1, y1 = tri[0]
    x2, y2 = tri[1]
    x3, y3 = tri[2]

    # Barycentric coordinates
    denom = (y2 - y3) * (x1 - x3) + (x3 - x2) * (y1 - y3)
    # avoid divide-by-zero in degenerate triangles
    if denom == 0:
        return np.zeros_like(X, dtype=bool)

    a = ((y2 - y3) * (X - x3) + (x3 - x2) * (Y - y3)) / denom
    b = ((y3 - y1) * (X - x3) + (x1 - x3) * (Y - y3)) / denom
    c = 1.0 - a - b

    return (a >= 0) & (b >= 0) & (c >= 0)

def siemens_star(nobj, step_deg=15):
    # Triangle in (x,y) pixel coordinates
    tri = np.array([
        (1.5*nobj//16, nobj//2 - nobj//32),
        (1.5*nobj//16, nobj//2 + nobj//32),
        (nobj//2 - nobj//128, nobj//2),
    ], dtype=np.float32)

    # Pixel-center grid for rasterization (same coordinate system as tri)
    yy, xx = np.mgrid[0:nobj, 0:nobj]  # yy rows, xx cols

    star = np.zeros((nobj, nobj), dtype=np.float32)
    center = np.array([nobj/2, nobj/2], dtype=np.float32)

    for deg in range(0, 360, step_deg):
        theta = np.deg2rad(deg)
        tri_r = rotate_points(tri, theta, center)
        star += triangle_mask(xx, yy, tri_r).astype(np.float32)

    # normalize to [0,1] like your star*circ/255 (but without 255 scaling)
    star /= star.max() if star.max() else 1.0
    return star

star = siemens_star(nobj, step_deg=15)

# --- your ring "holes" mask, kept equivalent ---
x, y = np.meshgrid(np.arange(-nobj//2, nobj//2), np.arange(-nobj//2, nobj//2))
x = x / nobj * 2
y = y / nobj * 2
r2 = x**2 + y**2

circ = ((r2 > 0.355) | (r2 < 0.345))
circ &= ((r2 > 0.083) | (r2 < 0.08))
circ &= ((r2 > 0.0085) | (r2 < 0.008))

star = star * circ.astype(star.dtype)

# --- optional Gaussian smoothing via FFT (as you had) ---
v = np.arange(-nobj//2, nobj//2) / nobj
vx, vy = np.meshgrid(v, v)
gauss = np.exp(-5*(vx**2 + vy**2))

fu = np.fft.fftshift(np.fft.fftn(np.fft.fftshift(star)))
star = np.fft.fftshift(np.fft.ifftn(np.fft.fftshift(fu * gauss))).real

# --- projection (unchanged) ---
delta = 1 - xraylib.Refractive_Index_Re('Au', energy, 19.3)
beta  = xraylib.Refractive_Index_Im('Au', energy, 19.3)

thickness = 1e-6 / voxelsize  # pixels
u = star * (-delta + 1j*beta)  # note -delta
proj = u * thickness * voxelsize * 2*np.pi / wavelength
proj = proj[np.newaxis].astype(np.complex64)

proj = np.tile(proj,[ntheta,1,1])
mshow_complex(proj[0])

# Probe, upsample it to n=1024

In [ ]:
prb = cp.array(tifffile.imread('illumination_intensity.tif')*np.exp(1j*(tifffile.imread('illumination_phase.tif'))))
prb = ndimage.zoom(prb,n/prb.shape[-1]).astype('complex64')
mshow_polar(prb)

# Positions 

In [ ]:
cp.random.seed(3)
pos = (cp.random.random([ntheta,ndist,2])-0.5).astype('float32')*50

# Reconstruction class

In [ ]:
# Form reconstruction parameters
rargs = SimpleNamespace(
    ndist=ndist,
    ntheta=ntheta,
    energy=energy,        
    focusToDetectorDistance=focusToDetectorDistance,        
    z1=z1,       
    obj_dtype='complex64',    
    n = n,
    nz = n,
    nobj = nobj,
    nzobj = nobj,
    detector_pixelsize = detector_pixelsize,           
)
cl = Rec(rargs)


## Generate data
## $ d_j = |\mathcal{D}_{z_j}(\mathcal{D}_{z'_j}(p)\cdot e^{i\mathcal{S}_{r_j} ( \delta+i\beta) })|^2$, 
### where $\mathcal{S}_{r_j}$ is a shift+magnification for distance $j$ 
### $r_j$ - (x,y) sample positions at distance $j$ given in pixels for original scaling

In [ ]:
vars = {}
vars['proj'] = cp.array(proj)
vars['prb'] = cp.array(prb)
vars['pos'] = cp.array(pos)

data = cp.empty([ntheta,ndist,n,n],dtype='float32')
cl.gen_data(vars,data)

ref = cp.empty([n,n],dtype='float32')
cl.gen_ref(vars['prb'],ref)

mshow(data[0,0])
mshow(data[0,-1])
mshow(ref)

## make some bias in positions and the probe for the initial guess

In [ ]:
from cupyx.scipy.ndimage import gaussian_filter
cp.random.seed(10)
prb_init = gaussian_filter(prb, sigma=5)     # Gaussian blur
pos_err = (cp.random.random([ntheta,ndist,2])-0.5).astype('float32')*5
pos_init = pos+pos_err
mshow_pos(-pos_err)
mshow_polar(prb_init)

## normalize with mean (ref)

In [ ]:
mean_ref = cp.mean(ref)
data/=mean_ref
ref/=mean_ref

## Solving with the BH method:
### $ \argmin_{\delta,\beta,p,r}\|\sqrt{d_j} - |\mathcal{D}_{z_j}(\mathcal{D}_{z'_j}(p)\cdot e^{i\mathcal{S}_{r_j} ( \delta+i\beta) })|\|^2_2 +\lambda\|D_z (p) - \sqrt{d^r}\|_2^2 +\lambda_r\|\Delta (\delta+i\beta)\|_2^2$, 
### where $\mathcal{S}_{r_j}$ is a shift+magnification for distance $j$ 
### $r_j$ - (x,y) sample positions at distance $j$ given in pixels for original scaling
### $\lambda\|D_z (p) - \sqrt{d^r}\|_2^2$ - probe fit term
### $\lambda_r\|\Delta (\delta+i\beta)\|_2^2$ - regularization term


In [ ]:

rargs.niter = 1025 # number of iterations
rargs.vis_step = 128 # visualization step
rargs.err_step = 32  # error step
rargs.lam_prbfit = 1 # probe fit term
rargs.lam_reg = 0  # regularizagtion (if noisy)
rargs.rho = {'proj':1,'prb':1,'pos':0.05} # scaling variables

vars = {}
vars['prb'] = cp.array(prb_init)/cp.sqrt(mean_ref) ## init is also normalized
vars['pos'] = cp.array(pos_init)
vars['proj'] = cp.zeros([ntheta,nobj,nobj],dtype='complex64') # can be also estimated with e.g. multiPaganin
cl = Rec(rargs)    
cl.BH(data,ref,vars)



In [ ]:
mshow_complex(vars['proj'][0])